# SNIFF EXPERIMENT TUTORIAL (under construction)

# Objectives

This tutorial demonstrates how Interactional Motivation (IM) can drive active perception. 
Similar to the [Smell Experiment Tutorial](IM_Tutorial_02.ipynb), this tutorial uses IM to implement an "instinctive behavior" to approach targets that "smell good".

In this experiment, we endow the agent with additional actions that allow it to actively "sniff" its environment.

As before, the agent initially ignores the meaning of possible actions and sensory signals. 
We expect it to learn to use the "sniff" actions as active perception to avoid bumping into walls and to sense the direction of the target.

See the Interactional Motivation in AIRIS documentation at [interactional_motivation.md](interactional_motivation.md)

# Let's Define the possibilities of interaction

This agent can perform six actions in this environment: 
* `move_forward`
* `turn_left`
* `turn_right` 
* `sniff_front` 
* `sniff_left` 
* `sniff_right` 

Each action may yield five possible outcomes depending on the smell of the cell in front of the agent:

* `decrease`: the smell decreased or the agent sniffed a wall
* `stable`: the smell remained stable
* `increase`: the smell increased
* `bump` : the agent collided with a wall (only after `move_forward`)
* `eat`: the agent reached the target and ate it (only after `move_forward`)

In [2]:
# Actions
SNIFF_FRONT = 1
FORWARD = 0
SNIFF_LEFT = 2
SNIFF_RIGHT = 3
TURN_LEFT = 4
TURN_RIGHT = 5

# Outcomes
DECREASE = 0
STABLE = 1
INCREASE = 2
BUMP = 3
EAT = 4

# Create the agent

In [3]:
class CompositeInteraction:
    """A composite interaction is a tuple (pre_interaction, post_interaction) and a weight"""
    def __init__(self, pre_interaction, post_interaction):
        self.pre_interaction = pre_interaction
        self.post_interaction = post_interaction
        self.weight = 1
        self._step = 1

    def get_decision(self):
        """Return the flatten sequence of intermediary primitive interactions terminated with the final decision"""
        return f"{self.pre_interaction.sequence()}{self.post_interaction.get_decision()}"

    def get_actions(self):
        """Return the flat sequence of the decisions of this interaction as a string"""
        return f"{self.pre_interaction.get_actions()}{self.post_interaction.get_actions()}"

    def get_valence(self):
        """Return the valence of the pre_interaction plus the valence of the post_interaction"""
        return self.pre_interaction.get_valence() + self.post_interaction.get_valence()

    def reinforce(self):
        """Increment the composite interaction's weight"""
        self.weight += 1

    def key(self):
        """ The key to find this interaction in the dictionary is the string '<pre_interaction><post_interaction>'. """
        return f"({self.pre_interaction.key()},{self.post_interaction.key()})"

    def pre_key(self):
        """Return the key of the pre_interaction"""
        #if self.weight > confidence_threshold:
        return self.pre_interaction.key()
        #else:
        #return self.pre_interaction.pre_key()

    def __str__(self):
        """ Print the interaction in the Newick tree format (pre_interaction, post_interaction: valence) """
        return f"({self.pre_interaction}, {self.post_interaction}: {self.weight})"

    def __eq__(self, other):
        """ Interactions are equal if they have the same pre and post interactions """
        if isinstance(other, self.__class__):
            return (self.pre_interaction == other.pre_interaction) and (self.post_interaction == other.post_interaction)
        else:
            return False

    def get_length(self):
        """Return the length of the number of primitive interactions in this composite interaction"""
        return self.pre_interaction.get_length() + self.post_interaction.get_length()

    def increment(self, interaction, interactions):
        """Increment the step of the appropriate sub-interaction. Return the enacted interaction if it is over, or None if it is ongoing."""
        # First step 
        if self._step == 1:
            interaction = self.pre_interaction.increment(interaction, interactions)
            # Ongoing pre-interaction. Return None
            if interaction is None:
                return None
            # Pre-interaction succeeded. Increment the step and return None
            elif interaction == self.pre_interaction:
                self._step = 2
                return None
            # Pre-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                return interaction
        # Second step
        else:
            interaction = self.post_interaction.increment(interaction, interactions)
            # Ongoing post-interaction. Return None
            if interaction is None:
                return None
            # Post-interaction succeeded. Reset the step and return this interaction
            elif interaction == self.post_interaction:
                self._step = 1
                return self
            # Post-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                composite_interaction = CompositeInteraction(self.pre_interaction, interaction)
                if composite_interaction.key() not in interactions:
                    # Add the enacted composite interaction to memory
                    interactions[composite_interaction.key()] = composite_interaction
                    if trace:
                        print(f"Learning {composite_interaction}")
                    return composite_interaction
                else:
                    # Reinforce the existing composite interaction and return it
                    interactions[composite_interaction.key()].reinforce()
                    if trace:
                        print(f"Reinforcing {interactions[composite_interaction.key()]}")
                    return interactions[composite_interaction.key()]

    def current(self):
        """Return the current intended primitive interaction"""
        # Step 1: the current primitive interaction of the pre-interaction
        if self._step == 1:
            return self.pre_interaction.current()
        # Step 2: The current primitive interaction of the post-interaction
        else:
            return self.post_interaction.current()

    def sequence(self):
        """Return the flat sequence of primitive interactions of this composite interaction"""
        return f"{self.pre_interaction.sequence()}{self.post_interaction.sequence()}"

    def get_post_interactions(self):
        """Return the list of the hierarchy of the sub post_interactions"""
        return [self.post_interaction] + self.post_interaction.get_post_interactions()

In [4]:
class Interaction:
    """An interaction is a tuple (action, outcome) with a valence"""
    def __init__(self, _action, _outcome, _valence):
        self._action = _action
        self._outcome = _outcome
        self._valence = _valence
        self.weight = 10
        
    def get_action(self):
        """Return the action"""
        return self._action

    def get_actions(self):
        """Return the action as a string for compatibilty with CompositeInteraction"""
        return str(self._action)

    def get_decision(self):
        """Return the decision key"""
        return f"{self._action}"
        # return f"a{self._action}"

    def get_outcome(self):
        """Return the action"""
        return self._outcome

    def get_valence(self):
        """Return the action"""
        return self._valence

    def key(self):
        """ The key to find this interaction in the dictinary is the string '<action><outcome>'. """
        return f"{self._action}{self._outcome}"

    def pre_key(self):
        """Return the key. Used for compatibility with CompositeInteraction"""
        return ""  # self.key()

    def __str__(self):
        """ Print interaction in the form '<action><outcome:<valence>' for debug."""
        return f"{self._action}{self._outcome}:{self._valence}"

    def __eq__(self, other):
        """ Interactions are equal if they have the same key """
        if isinstance(other, self.__class__):
            return self.key() == other.key()
        else:
            return False

    def get_length(self):
        """The length of the sequence of this interaction"""
        return 1

    def increment(self, interaction, interactions):
        """Return the enacted interaction for compatibility with composite interactions"""
        return interaction

    def current(self):
        """Return itself for compatibility with composite interactions"""
        return self

    def sequence(self):
        """Return the key. Use for compatibility with composite interactions"""
        return self.key()

    def get_post_interactions(self):
        """Return the empty list for compatibility with composite interactions"""
        return []

In [5]:
# Display codes
SNIFF_EMPTY = 1
SNIFF_WALL = 2
BUMPING = 3

TARGET = 2

trace = True
# Maximum length of intended sequence
max_length = 10
# Minimum weight of intended sequence
min_weight = 1

In [6]:
import pandas as pd

class Agent:
    def __init__(self, _interactions):
        """ Initialize our agent """
        self._interactions = {interaction.key(): interaction for interaction in _interactions}
        self._primitive_intended_interaction = self._interactions[f"{SNIFF_FRONT}{STABLE}"]
        self._intended_interaction = None

        # The context
        self._penultimate_interaction = None
        self._previous_interaction = None
        self._last_interaction = None
        self._penultimate_composite_interaction = None
        self._previous_composite_interaction = None
        self._last_composite_interaction = None
        
        # Prepare the dataframe of proposed interactions
        default_interactions = [interaction for interaction in _interactions if interaction.get_outcome() == STABLE]
        data = {'activated': [""] * len(default_interactions),
                'weight': [0] * len(default_interactions),
                'actions': [i.get_actions() for i in default_interactions],
                'intention': [i.key() for i in default_interactions],
                'valence': [i.get_valence() for i in default_interactions],
                'decision': [i.get_decision() for i in default_interactions],
                'length': [1] * len(default_interactions),
                'pre': [""] * len(default_interactions)} 
        self._default_df = pd.DataFrame(data)
        self.proposed_df = None
        self.decision_df = None
        self.clear = True # Used to clear the display after the enacted interaction

    def action(self, _outcome):
        """Implement the agent's policy"""
        # Trace the previous cycle
        primitive_enacted_interaction = self._interactions[f"{self._primitive_intended_interaction.get_action()}{_outcome}"]
        if trace:
            print(
            f"Action: {self._primitive_intended_interaction.get_action()}, Prediction: {self._primitive_intended_interaction.get_outcome()}, "
            f"Outcome: {_outcome}, Prediction_correct: {self._primitive_intended_interaction.get_outcome() == _outcome}, "
            f"Valence: {primitive_enacted_interaction.get_valence()}")

        # Follow up the enaction
        if self._intended_interaction is None: # First interaction cycle
            enacted_interaction = primitive_enacted_interaction
        else:
            enacted_interaction = self._intended_interaction.increment(primitive_enacted_interaction, self._interactions)

        # If the intended interaction is over (completely enacted or aborted)
        if enacted_interaction is None:
            self.clear = False
        else:
            self.clear = True
            # Memorize the context
            self._penultimate_composite_interaction = self._previous_composite_interaction
            self._previous_composite_interaction = self._last_composite_interaction
            self._penultimate_interaction = self._previous_interaction
            self._previous_interaction = self._last_interaction
            self._last_interaction = enacted_interaction
            # Call the learning mechanism
            self.learn(enacted_interaction)
            # Create the proposed dataframe
            self.create_proposed_df()
            self.aggregate_propositions()
            # Decide the next enaction
            self.decide()

        # Return the next primitive action
        self._primitive_intended_interaction = self._intended_interaction.current()
        return self._primitive_intended_interaction.get_action()
        
    def learn(self, enacted_interaction):
        """Learn the composite interactions"""
        # First level of composite interactions
        self._last_composite_interaction = self.learn_composite_interaction(self._previous_interaction, enacted_interaction)
        # Second level of composite interactions
        self.learn_composite_interaction(self._previous_composite_interaction, enacted_interaction)
        self.learn_composite_interaction(self._penultimate_interaction, self._last_composite_interaction)

        # Higher level composite interaction made of two composite interactions
        if self._last_composite_interaction is not None:
            self.learn_composite_interaction(self._penultimate_composite_interaction, self._last_composite_interaction)

    def learn_composite_interaction(self, pre_interaction, post_interaction):
        """Record or reinforce the composite interaction made of (pre_interaction, post_interaction)"""
        if pre_interaction is None:
            return None
        else:
            # If the pre-interaction exists
            composite_interaction = CompositeInteraction(pre_interaction, post_interaction)
            if composite_interaction.key() not in self._interactions:
                # Add the composite interaction to memory
                self._interactions[composite_interaction.key()] = composite_interaction
                if trace:
                    print(f"Learning {composite_interaction}")
                return composite_interaction
            else:
                # Reinforce the existing composite interaction and return it
                self._interactions[composite_interaction.key()].reinforce()
                if trace:
                    print(f"Reinforcing {self._interactions[composite_interaction.key()]}")
                return self._interactions[composite_interaction.key()]

    def create_proposed_df(self):
        """Create the proposed dataframe from the activated interactions"""
        # The list of activated interactions that match the current context
        activated_interactions = [i for i in self._interactions.values() if i.get_length() > 1 
                                  and i.pre_interaction in self._last_composite_interaction.get_post_interactions()]
        data = {'activated': [i.key() for i in activated_interactions],
                'weight': [i.weight for i in activated_interactions],
                'actions': [i.post_interaction.get_actions() for i in activated_interactions],
                'intention': [i.post_interaction.key() for i in activated_interactions],
                'valence': [i.post_interaction.get_valence() for i in activated_interactions],
                'decision': [i.post_interaction.get_decision() for i in activated_interactions],
                'pre': [i.post_interaction.pre_key() for i in activated_interactions],
                'length': [i.post_interaction.get_length() for i in activated_interactions],
                }
        activated_df = pd.DataFrame(data).astype(self._default_df.dtypes)  # Force the same types in case activated_df is empty

        # Create the proposed dataframe
        self.proposed_df = pd.concat([self._default_df, activated_df], ignore_index=True).sort_values(by='decision', ascending=True).reset_index(drop=True)

        # Calculate the proclivity of each proposition
        self.proposed_df['proclivity'] = self.proposed_df['weight'] * self.proposed_df['valence']

        # Compute the probability of each propositions
        # self.proposed_df['probability'] = self.proposed_df['weight'] / self.proposed_df.groupby('actions')['weight'].transform('sum')
        # self.proposed_df['probability'] = self.proposed_df.groupby('intention')['weight'].transform('sum') / self.proposed_df.groupby('actions')['weight'].transform('sum')

    def aggregate_propositions(self):
        """Aggregate the proclivity"""
        # Aggregate the proclivity for each decision
        grouped_df = self.proposed_df.groupby('decision').agg({'proclivity': 'sum', 'actions': 'first', # 'action': 'first', 
                                                               'length': 'first', 'intention': 'first', 'pre': 'first'}).reset_index()
        # For each proposed composite decision 
        for index, proposed in grouped_df[grouped_df['length'] > 1].iterrows():
            # print(f"Index {index}, actions {proposition['actions']}, intention {proposition['intention']}")
            # Find shorter decisions that start with the same sequence 
            for _, shorter in self.proposed_df[self.proposed_df.apply(lambda row: proposed['actions'].startswith(row['actions']) 
                                                                      and row['length'] < proposed['length'], axis=1)].iterrows():
                # Add the proclivity of the shorter decisions
                grouped_df.loc[index, 'proclivity'] += shorter['proclivity']
                # print(f"Decision {proposed['decision']} recieves {shorter['proclivity']} from shorter {shorter['intention']}")

        # Remove the intentions that are insufficiently reinforced
        grouped_df = grouped_df[grouped_df['intention'].apply(lambda x: self._interactions[x].weight) >= min_weight]
        # Remove the intentions that are too long
        grouped_df = grouped_df[grouped_df['intention'].apply(lambda x: self._interactions[x].get_length()) <= max_length]
        
        # Sort by descending proclivity
        self.decision_df = grouped_df.sort_values(by=['proclivity', 'decision'], ascending=[False, True]).reset_index(drop=True)

    def decide(self):
        """Selects the intended_interaction at the top of the proposed dataframe"""
        # The intended interaction is in the first row because it has been sorted by descending proclivity
        intended_interaction_key = self.decision_df.loc[0, 'intention']
        if trace:
            print("Intention:", intended_interaction_key)
        self._intended_interaction = self._interactions[intended_interaction_key]

# Create the environment 

## Define the grid

In [7]:
import numpy as np

# 0: Empty cell. 1: Wall
grid = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
])

## Define the environment class

In [8]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgb, to_rgba
from ipywidgets import Button, HBox, VBox, Output, widgets
from IPython.display import display

agent_color = "#1976D2"
# Environment colors: Empty, Wall, Target
environment_colors = ["#D6D6D6", '#5C946E', "#E365C1"]
# Interaction colors with transparency: None, Feel_empty, Feel_wall, Bump
interaction_colors = ['#00000000', '#FAE2DBFF', '#535865FF', "#F93943FF"]
# Directions
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
MAX_SMELL = 100

class Environment():
    def __init__(self, position, direction):
        self.env_grid = grid.copy()
        self.int_grid = np.zeros(grid.shape, dtype=int)
        self.position = np.array(position)  # Using NumPy array of shape (2)
        self.direction = direction
        self.env_colors = np.array([to_rgb(c) for c in environment_colors])
        self.int_colors = np.array([to_rgba(c) for c in interaction_colors])
        brighter_target =  to_hex(self.env_colors[2] + (1 - self.env_colors[2]) * 0.2)
        self.cmap_smell = LinearSegmentedColormap.from_list("smell_gradiant", [brighter_target, "#ffffff"])
        self.marker_size = 400
        self.marker_map = {LEFT: '<', DOWN: 'v', RIGHT: '>', UP: '^'}
        self.marker_color = agent_color
        self.directions = np.array([
            [0, -1],  # Left
            [1, 0],   # Down
            [0, 1],   # Right
            [-1, 0]   # Up
        ])
        self.smell_grid = np.full(self.env_grid.shape, MAX_SMELL)
        self.fig = None  
        # --- Build the smell graph ---
        self.G = nx.Graph()
        # Connect all adjacent (non-wall) cells
        for r in range(self.env_grid.shape[0]):
            for c in range(self.env_grid.shape[1]):
                if self.env_grid[r, c] == 1:
                    continue  # skip walls
                for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < self.env_grid.shape[0] and 0 <= nc < self.env_grid.shape[1] and self.env_grid[nr, nc] != 1:
                        self.G.add_edge((r, c), (nr, nc), weight=1)

    def outcome(self, action):
        """Update the grid. Return the outcome of the action."""
        result = STABLE

        front_position = self.position + self.directions[self.direction]
        left_position = self.position + self.directions[(self.direction + 1) % 4]
        right_position = self.position + self.directions[self.direction - 1]
        position_smell = self.smell_grid[tuple(self.position)]
        front_smell = self.smell_grid[tuple(front_position)]
        left_smell = self.smell_grid[tuple(left_position)]
        right_smell = self.smell_grid[tuple(right_position)]

        if action == FORWARD:  
            if self.env_grid[tuple(front_position)] in [0, TARGET]:  # Don't bump in targets
                self.position[:] = front_position
                new_front_smell = self.smell_grid[tuple(self.position + self.directions[self.direction])]
                result = STABLE + (new_front_smell < front_smell) - (new_front_smell > front_smell)
            else:
                result = BUMP
                self.int_grid[tuple(front_position)] = BUMPING
            # Eat only when move forward
            if self.env_grid[tuple(self.position)] == TARGET:
                self.clear_target(tuple(self.position))
                result = EAT
        
        elif action == TURN_RIGHT:
            result = STABLE + (right_smell < front_smell) - (right_smell > front_smell)
            self.direction = (self.direction + 3) % 4
        
        elif action == TURN_LEFT:
            result = STABLE + (left_smell < front_smell) - (left_smell > front_smell)
            self.direction = (self.direction + 1) % 4
        
        elif action == SNIFF_FRONT:
            if self.env_grid[tuple(front_position)] == 1:
                result = DECREASE # BUMP
                self.int_grid[tuple(front_position)] = SNIFF_WALL
            else:
                self.int_grid[tuple(front_position)] = SNIFF_EMPTY
                result = STABLE + (front_smell < position_smell) - (front_smell > position_smell)
        
        elif action == SNIFF_LEFT:
            if self.env_grid[tuple(left_position)] == 1:
                result = DECREASE # BUMP
                self.int_grid[tuple(left_position)] = SNIFF_WALL
            else:
                result = STABLE + (left_smell < position_smell) - (left_smell > position_smell)
                self.int_grid[tuple(left_position)] = SNIFF_EMPTY
        
        elif action == SNIFF_RIGHT:
            if self.env_grid[tuple(right_position)] == 1:
                result = DECREASE # BUMP
                self.int_grid[tuple(right_position)] = SNIFF_WALL
            else:
                result = STABLE + (right_smell < position_smell) - (right_smell > position_smell)
                self.int_grid[tuple(right_position)] = SNIFF_EMPTY

        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
    
    def display(self):
        """Display the grid in the notebook"""
        out.clear_output(wait=True)
        with out:
            # ChatGPT recommends closing and recreating the figure
            plt.close(self.fig)
            self.fig, ax = plt.subplots()
            # Display the environment
            ax.imshow(self.env_colors[self.env_grid])
            # Display the agent
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            # Display the smell in empty cells if any 
            if self.smell_grid.min() < MAX_SMELL:
                masked_smell = np.ma.masked_where(self.env_grid > 0, self.smell_grid)
                ax.imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))
            # Display the interactions
            ax.imshow(self.int_colors[self.int_grid])
            # Display the step
            ax.text(self.env_grid.shape[1]-1.5, 0.2, f"{step:>3}", fontsize=12, color='White')
            plt.show()
            cid = self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        
    def on_click(self, event):
        """Add or remove a target when the user clicks on the grid in widget mode"""
        if event.inaxes is self.fig.axes[0] and event.button == 1 and event.xdata is not None:
            position = (round(event.ydata), round(event.xdata))
            if self.env_grid[position] == 0:
                self.add_target(position)
            elif self.env_grid[position] == TARGET:
                self.clear_target(position)
            # Redrawing the axes renders faster than calling self.display()
            self.display()
            # self.fig.axes[0].imshow(self.int_colors[self.int_grid])
            # masked_smell = np.ma.masked_where(self.smell_grid == MAX_SMELL, self.smell_grid)
            # self.fig.axes[0].imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))

    def save(self, step):
        """Save the display as a PNG file"""
        if "save_dir" in globals():
            fig, ax = plt.subplots()
            ax.set_xticks([])
            ax.set_yticks([])
            ax.axis('off')
            # Display the environment
            ax.imshow(self.env_colors[self.env_grid])
            # Display the agent
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            # Display the smell in empty cells if any
            if self.smell_grid.min() < MAX_SMELL:
                masked_smell = np.ma.masked_where(self.env_grid > 0, self.smell_grid)
                ax.imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))
            # Display the interactions
            ax.imshow(self.int_colors[self.int_grid])
            # Display the step
            ax.text(self.env_grid.shape[1]-1.5, 0.2, f"{step:>3}", fontsize=12, color='White')
            plt.savefig(f"{save_dir}/{step:03}.png", bbox_inches='tight', pad_inches=0, transparent=True)
            plt.close(fig)
    
    def clear(self, clear):
        """Clear the grid display"""
        if clear:
            self.int_grid[:, :] =  0  

    def add_target(self, position):
        """Insert a new target at position (l, c)"""
        self.env_grid[position] = TARGET
        self.smell_map()

    def clear_target(self, position):
        """Remove target at position (l, c)"""
        self.env_grid[position] = 0
        self.smell_map()

    def smell_map(self):
        """Construct the smell map using the Dijkstra algorithm"""
        # --- Identify target cells ---
        targets = [(r, c) for r in range(self.env_grid.shape[0]) for c in range(self.env_grid.shape[1]) if self.env_grid[r, c] == TARGET]        
        # --- Compute shortest distances from each cell to the nearest target ---
        self.smell_grid[:, :] = MAX_SMELL
        for target in targets:
            lengths = nx.single_source_dijkstra_path_length(self.G, target)
            for (r, c), dist in lengths.items():
                self.smell_grid[r, c] = min(self.smell_grid[r, c], dist)


# Demonstrate the agent

## Initialize the interactions 

In [9]:
interactions = [
    Interaction(FORWARD,STABLE,2),
    Interaction(FORWARD,BUMP,-10),
    Interaction(TURN_LEFT,STABLE,-3),
    Interaction(TURN_LEFT,BUMP,-3),
    Interaction(TURN_RIGHT,STABLE,-3),
    Interaction(TURN_RIGHT,BUMP,-3),
    Interaction(SNIFF_FRONT,STABLE,-1),
    Interaction(SNIFF_FRONT,BUMP,-1),
    Interaction(SNIFF_LEFT,STABLE,-1),
    Interaction(SNIFF_LEFT,BUMP,-1),
    Interaction(SNIFF_RIGHT,STABLE,-1),
    Interaction(SNIFF_RIGHT,BUMP,-1),
    
    Interaction(SNIFF_FRONT,DECREASE,-1),
    Interaction(SNIFF_FRONT,INCREASE,-1),
    Interaction(SNIFF_LEFT,DECREASE,-1),
    Interaction(SNIFF_LEFT,INCREASE,-1),
    Interaction(SNIFF_RIGHT,DECREASE,-1),
    Interaction(SNIFF_RIGHT,INCREASE,-1),
    Interaction(FORWARD, INCREASE, 5),
    Interaction(FORWARD, DECREASE, -5),
    Interaction(FORWARD, EAT, 5),
    Interaction(TURN_LEFT, INCREASE, -1),
    Interaction(TURN_LEFT, DECREASE, -5),
    Interaction(TURN_LEFT, EAT, 1),
    Interaction(TURN_RIGHT, INCREASE, -1),
    Interaction(TURN_RIGHT, DECREASE, -5),
    Interaction(TURN_RIGHT, EAT, 1),
]

# Run the agent in a loop

Choose the steps and the positions of new targets to insert in `target_steps` et `target_positions`.

In [10]:
# The steps when new targets are inserted
target_steps = [40, 80, 120, 160, 200, 240, 280, 320]
# For each insertion step, give the position of the target (line, column). The keys must correspond to the steps above.
target_positions = {0:(5,2), 40:(6,6), 80:(3,2), 120:(2,3), 160: (1, 1), 200:(4,4), 240:(2,3), 280:(3,1), 320:(2,10)}

Initialize the simulation

In [25]:
# Deactivate trace
trace = False

# Instanciate the environment
e = Environment([6, 4], UP)
# e = Environment([2, 10], UP)
# Initialize the agent
a = Agent(interactions)
outcome = 0

# Display
out = Output()
step = 0
e.display()
display(out)


Output()

In [26]:
for step in range(400):
    if step in target_steps:
        e.add_target(target_positions[step])
    action = a.action(outcome)
    e.display()
    plt.pause(0.01) # increase to play more slowly but do not remove or it flickers
    e.save(step)  # Save a screenshot 
    e.clear(True)
    outcome = e.outcome(action)

The sniffing interactions are materialized as flashing squares. 
The target is represented by a magenta cell, and its smell by a magenta gradient.

Observe that, after catching a few targets, the agent learns to reach the target efficiently by following a stair-case trajectory until it aligns itself with the target moves straigt to it. 

# Try different configurations


Try to modify the valence of interactions, the shape of the environment, or the position and time when targets are inserted. 

The experiment is deterministic, so two runs in the same configuration will result in the exact same behavior. 
The learned behavior nonetheless depends on the particular settings.
For example, if you give a positive valence to sniffing or bumping, the agent will keep sniffing and bumping franticaly. 

When two actions receive identical activation values, the schema mechanism selects the one with the lowest code. 
As a result, the default ordering of actions has a marginal influence on behavior. 
You can modify the code assigned to the actions to modify this default ordering.

If the environment is relatively open, and the valence for moving forward is positive, the agent will tend to move forward without sniffing at the risk of bumping into walls. 

If there are many walls and if the valence of bumping is strongly negative then the agent will tend to sniff to avoid bumping into walls. 

If you wish to record your own experiment as a GIF animation, use the code below to define the variable `save_dir`. 
The screenshots will be saved in your specified directory.

In [27]:
# The sub-directory to save the images. Use "." to save the images in the same directory as this notebook
# save_dir = "sav"  # "."

# Examples

## 1. Example run where the agent starts in the large open area

![](img/03_movie_1.gif)

When there is no target, the agent just enjoys moving forward. 
It does not learn to sniff to avoid bumping into walls. 
The last bumping is on Step 396.

## 2. Example run where the agent starts in the small area

![](img/03_movie_2.gif)

The agent practices sniffing to avoid bumping into walls. 
The last bump is on Step 70. 
After learning the sniffing behavior, the agent does not bump into walls anymore. 

## 3. Example run where the agent patiently waits for new target

In [21]:
interactions = [
    Interaction(FORWARD,STABLE,-1),
    Interaction(FORWARD,BUMP,-10),
    Interaction(TURN_LEFT,STABLE,-3),
    Interaction(TURN_LEFT,BUMP,-3),
    Interaction(TURN_RIGHT,STABLE,-3),
    Interaction(TURN_RIGHT,BUMP,-3),
    Interaction(SNIFF_FRONT,STABLE,1),
    Interaction(SNIFF_FRONT,BUMP,-1),
    Interaction(SNIFF_LEFT,STABLE,-1),
    Interaction(SNIFF_LEFT,BUMP,-1),
    Interaction(SNIFF_RIGHT,STABLE,-1),
    Interaction(SNIFF_RIGHT,BUMP,-1),
    
    Interaction(SNIFF_FRONT,DECREASE,-1),
    Interaction(SNIFF_FRONT,INCREASE,-1),
    Interaction(SNIFF_LEFT,DECREASE,-1),
    Interaction(SNIFF_LEFT,INCREASE,-1),
    Interaction(SNIFF_RIGHT,DECREASE,-1),
    Interaction(SNIFF_RIGHT,INCREASE,-1),
    Interaction(FORWARD, INCREASE, 5),
    Interaction(FORWARD, DECREASE, -5),
    Interaction(FORWARD, EAT, 5),
    Interaction(TURN_LEFT, INCREASE, -1),
    Interaction(TURN_LEFT, DECREASE, -5),
    Interaction(TURN_LEFT, EAT, 1),
    Interaction(TURN_RIGHT, INCREASE, -1),
    Interaction(TURN_RIGHT, DECREASE, -5),
    Interaction(TURN_RIGHT, EAT, 1),
]
# The steps when new targets are inserted
target_steps = [0, 40, 80, 120, 160, 200, 240, 280, 320]
# For each insertion step, give the position of the target (line, column). The keys must correspond to the steps above.
target_positions = {0:(5,2), 40:(6,6), 80:(3,2), 120:(2,3), 160: (1, 1), 200:(4,4), 240:(2,3), 280:(3,1), 320:(2,10)}

![](img/03_movie_3.gif)

This behaviors comes from the positive valence given to the interaction `sniffing stable smell in front`.